<a href="https://colab.research.google.com/github/sweet-cross/cross_notebooks/blob/main/notebooks/workshop_20260309_assumptions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
if 'google.colab' in sys.modules:
    # Installs the repo as a package, along with its dependencies
    !pip install git+https://github.com/sweet-cross/cross_notebooks.git

# CROSS Assumptions

This notebook uses the [CrossContracts](https://sweet-cross.github.io/crosscontract/)
package and the *CrossClient* therein to interact with the inputs assumptions 
impose by the CROSS 2025 scenarios.

To be able to interact with the assumptions using the CrossClient you need an 
account on the Cross platform. You can create an account here: (Register Account)[https://sweet-cross.ch/register]

Alternatively, all data are also distributed through (CROSSDat)[https://sweet-cross.ch/crossdat].

## Packages and options

In [2]:
from crosscontract import CrossClient
import getpass

## Connecting to CROSS

To connect to CROSS and to use the API, you use the CrossClient given your username
and password. You can either use your username or mail address to connect.

In [3]:
username = input("Enter your username or mail address: ")
password = getpass.getpass("Enter your password: ")
my_client = CrossClient(username=username, password=password)

## Finding contracts

CROSS relies on data contracts. A data contract specifies the schema of the data. The 
schema defines the fields, i.e., the columns, of the data together with a description
of the column, the type of data, and possibly some constraints (e.g., minimum value of zero).

A simple contract looks like:

```yaml
name: contract_gdp
title: Gross Domestic Product (GDP)
description: |
  Gross Domestic Product (GDP) by country and years.
tableschema:
  fields:
    - name: country
      type: string
      title: Country Name
      description: Name of the country
      constraints:
        required: true
        maxLength: 100
    - name: year
      type: integer
      title: Year
      description: Year of the GDP data
      constraints:
        required: true
        minimum: 2000
        maximum: 2050
    - name: gdp
      type: number
      title: GDP Value
      description: Gross Domestic Product value in USD
      constraints:
        required: true
        minimum: 0
```




Getting data generally follows a two step procedure:
1. Get the contract given the contract identifier
2. Given the the contract, get the associated data

Thus, you must know the name of the contract to access the data. The easiest way
to find the contract name, is to get an overview of all contract. Contracts for 
assumptions start usually with *scenass*. We can list all contract using the 
*overview* method which provides a pandas dataframe:

In [4]:
df_assumptions = (
    my_client.contracts.overview()
    .query("name.str.startswith('scenass')")
    .query("status == 'Active'")
    [["name", "title", "description"]]
)
df_assumptions

,name,title,description
0,scenass_cost_generation_technologies,Investment cost of generation technologies,Investment cost data for electricity generatio...
1,scenass_cost_heating_technologies,Investment cost of heating technologies,Investment cost data for heating technologies ...
3,scenass_aviation_fuel_demand,Aviation Fuel Demand,Aviation fuel demand data used as assumptions ...
4,scenass_biomass_potential,Biomass Potential,Biomass potentials by biomass type used as ass...
5,scenass_cost_storage_technologies,Investment cost of storage technologies,Investment cost data for storage technologies\...
6,scenass_electric_appliances_useful_energy_demand,Useful energy demand of electric appliances,Electricity demand from electric appliances an...
7,scenass_households,Number of Households,Household data used as assumptions for scenari...
8,scenass_energy_reference_area,Energy Reference Area,"Energy reference area by sector, year, buildin..."
9,scenass_freight_transport_useful_energy_demand,Useful energy demand in freight transport,Useful energy demand in freight transport by t...
10,scenass_gdp,Gross Domestic Product (GDP),GDP data used as assumptions for scenario runs.


## Getting and inspecting contracts


Getting a contract simply uses the name and the `client.contract.get()` method.

In [5]:
gen_cost = my_client.contracts.get("scenass_cost_generation_technologies")
gen_cost

ContractResource(name=scenass_cost_generation_technologies, status=Active)

Since we work with contracts that are stored on the CROSS platform, we receive a 
*ContractResource* which is read-only and represents a CrossContract and its data
stored on the platform. 

To inspect the actual contract, use the *contract* property of the resource, i.e., the
[CrossContract](https://sweet-cross.github.io/crosscontract/contracts/)

In [6]:
gen_cost_contract = gen_cost.contract
print(f"Contract Name: {gen_cost_contract.name}")
for field in gen_cost_contract.tableschema.fields:
    print(f"Name: {field.name}")
    print(f"Title: {field.title}")
    print(f"Description: {field.description.strip()}")
    print(f"Type: {field.type}")
    print(f"Constraints: {field.constraints}")
    print("-" * 40)

Contract Name: scenass_cost_generation_technologies
Name: scenario_group
Title: Scenario Group
Description: Name of the scenario group this data is associated with
Type: string
Constraints: required=False unique=False pattern=None minLength=None maxLength=None enum=None
----------------------------------------
Name: scenario_name
Title: Scenario Name
Description: Name of the specific scenario this data is associated with
Type: string
Constraints: required=False unique=False pattern=None minLength=None maxLength=None enum=None
----------------------------------------
Name: scenario_variant
Title: Scenario Variant
Description: Variant of the scenario, if applicable
Type: string
Constraints: required=False unique=False pattern=None minLength=None maxLength=None enum=None
----------------------------------------
Name: technology
Title: Generation Technology
Description: Generation Technology used
Type: string
Constraints: required=True unique=False pattern=None minLength=None maxLength=Non

In general, all assumptions (and results) follow a similar structure. A scenario
is defined by:
1. scenario_group: E.g. cross2025
2. scenario_name: The name of the scenario
3. scenario_variant: Allows for variations with in a given scenario (e.g., WACC sensitivity)

A table for a scenario assumption or results therefore usually looks like:

| scenario_group | scenario_name | scenario_variant | dimension_1 | ... | unit | value |
| --- | --- | --- | --- | --- | --- | --- |

The dimensions usually refer to the dimensions as fixed in the [CROSS data model](https://sweet-cross.ch/data-model). The
possible values and there description can be either retreived through the API (contracts start with "dim")
or inspected in the Git repo: https://sweet-cross.github.io/data-model/

## Getting the data

To get the data stored at CROSS, you use the *ContractResource* just obtained and 
its `get_data()` method. It returns the data as pandas dataframe:

In [7]:
df_gen_cost = gen_cost.get_data()
print("Available scenario groups:")
for group in list(df_gen_cost['scenario_group'].unique()):
    print(f" - {group}")
print("Available scenarios: ")
for scenario in list(df_gen_cost['scenario_name'].unique()):
    print(f" - {scenario}")
print("Available scenario variants: ")
for variant in list(df_gen_cost['scenario_variant'].unique()):
    print(f" - {variant}")
df_gen_cost.head()

Available scenario groups:
 - cross202506
Available scenarios: 
 - abroad-res-full
 - abroad-res-lim
 - abroad-nores-full
 - abroad-nores-lim
 - domestic-res-full
 - domestic-res-lim
 - domestic-nores-full
 - domestic-nores-lim
Available scenario variants: 
 - low
 - reference
 - high


,scenario_group,scenario_name,scenario_variant,technology,size,unit,year,value
0,cross202506,abroad-res-full,low,hydro_ror,< 10MW,CHF/kWe,2020,6160.0
1,cross202506,abroad-res-full,reference,hydro_ror,< 10MW,CHF/kWe,2020,9930.0
2,cross202506,abroad-res-full,high,hydro_ror,< 10MW,CHF/kWe,2020,13700.0
3,cross202506,abroad-res-full,low,hydro_ror,< 10MW,CHF/kWe,2035,4201.0
4,cross202506,abroad-res-full,reference,hydro_ror,< 10MW,CHF/kWe,2035,6772.0


Currently, we allow for limited filtering and querying of the data. E.g., to get the 
data only for the *reference* scenario variant:

In [8]:
df_gen_cost = gen_cost.get_data(filters={"scenario_variant": "reference"})
df_gen_cost.head()

,scenario_group,scenario_name,scenario_variant,technology,size,unit,year,value
0,cross202506,abroad-res-full,reference,hydro_ror,< 10MW,CHF/kWe,2020,9930.0
1,cross202506,abroad-res-full,reference,hydro_ror,< 10MW,CHF/kWe,2035,6772.0
2,cross202506,abroad-res-full,reference,hydro_ror,< 10MW,CHF/kWe,2050,6337.0
3,cross202506,abroad-res-full,reference,wind_on,,CHF/kWe,2020,2500.0
4,cross202506,abroad-res-full,reference,wind_on,,CHF/kWe,2035,2115.0


## All in one :)


If you are interested in only getting the data into a dataframe, you can simply chain the
previous steps:

In [9]:
df_gen_cost = (
    my_client
    .contracts
    .get("scenass_cost_generation_technologies")
    .get_data()
)
df_gen_cost.head()

,scenario_group,scenario_name,scenario_variant,technology,size,unit,year,value
0,cross202506,abroad-res-full,low,hydro_ror,< 10MW,CHF/kWe,2020,6160.0
1,cross202506,abroad-res-full,reference,hydro_ror,< 10MW,CHF/kWe,2020,9930.0
2,cross202506,abroad-res-full,high,hydro_ror,< 10MW,CHF/kWe,2020,13700.0
3,cross202506,abroad-res-full,low,hydro_ror,< 10MW,CHF/kWe,2035,4201.0
4,cross202506,abroad-res-full,reference,hydro_ror,< 10MW,CHF/kWe,2035,6772.0
